# 블랙아이스 생성 사전 조건 분석 (Pre-conditions Analysis)
블랙아이스는 발생 당일의 상태뿐만 아니라, **발생 전날 밤부터 새벽까지의 기상 조건**에 의해 형성됩니다. 
이 노트북은 기상청(KMA) 데이터를 활용하여 블랙아이스가 관측된 시점(2월 10일, 12일 오전) 이전 12~18시간 동안의 기상 변화(기온, 습도, 이슬점 등)를 역추적하여 생성 조건을 분석합니다.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

plt.rcdefaults() # 이전에 캐시된 한글 폰트 설정 제거

# 기상청 데이터 로드 함수 (JSON 파싱 및 데이터프레임 변환)
def load_weather_data(date_str):
    weather_file = Path(f'weather/kma_data_{date_str}.json')
    if not weather_file.exists(): return pd.DataFrame()
    with open(weather_file, 'r', encoding='utf-8') as f:
        weather_json = json.load(f)
    
    records = []
    for point_data in weather_json['collected_data']:
        for line in point_data['data'].split('\n'):
            if line.startswith('#') or not line.strip(): continue
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 6:
                records.append({
                    'datetime': datetime.strptime(parts[0], '%Y%m%d%H%M'),
                    'ta': float(parts[1]), 'hm': float(parts[2]),
                    'td': float(parts[3]), 'ws': float(parts[4]), 'rn': float(parts[5])
                })
    df = pd.DataFrame(records)
    # 동일 시간대의 지역 데이터를 평균내어 단일 시계열로 변환
    if not df.empty:
        df = df.groupby('datetime').mean().reset_index()
    return df

# 2월 9일~10일 데이터 결합
df_10 = pd.concat([load_weather_data('2026-02-09'), load_weather_data('2026-02-10')])
# 2월 11일~12일 데이터 결합
df_12 = pd.concat([load_weather_data('2026-02-11'), load_weather_data('2026-02-12')])

# 2월 10일 블랙아이스 분석 기간 (2월 9일 16:00 ~ 2월 10일 10:00)
mask_10 = (df_10['datetime'] >= '2026-02-09 16:00') & (df_10['datetime'] <= '2026-02-10 10:00')
df_analysis_10 = df_10[mask_10].copy()

# 2월 12일 블랙아이스 분석 기간 (2월 11일 16:00 ~ 2월 12일 10:00)
mask_12 = (df_12['datetime'] >= '2026-02-11 16:00') & (df_12['datetime'] <= '2026-02-12 10:00')
df_analysis_12 = df_12[mask_12].copy()

df_measure = pd.read_csv('measurements.csv')
df_measure_10 = df_measure[(df_measure['date'] == '2026-02-10') & (df_measure['black_ice_status'] == 'occurred')]

df_analysis_10.head()

### 1. 시계열 분석 (Time Series Analysis): 기온과 이슬점의 교차 현상
블랙아이스의 주 원인인 '서리(Frost)'는 대기 기온(ta)이 0도 아래로 떨어지는 동시에 이슬점(td) 근처에 머물 때 대기 중의 수분이 노면에 얼어붙어 발생합니다.
발생 전날 오후부터 다음 날 아침까지 기온이 어떻게 떨어지는지 흐름을 추적합니다.

🔗 **관련 자료:** [파이썬 시계열 데이터 시각화 (선 그래프 기초)](https://wikidocs.net/92071)

In [ ]:
def plot_time_series(df, title_date):
    if df.empty: return
    plt.figure(figsize=(12, 5))
    
    # 기온과 이슬점 시계열 선 그래프 (Line Plot)
    sns.lineplot(data=df, x='datetime', y='ta', label='Air Temp', color='red', linewidth=2)
    sns.lineplot(data=df, x='datetime', y='td', label='Dew Point', color='blue', linestyle='--', linewidth=2)
    
    # 0도 기준선
    plt.axhline(0, color='gray', linestyle=':', alpha=0.7)
    
    # 블랙아이스 발생 추정 시간대 (아침 8~10시) 주황색 배경으로 하이라이트
    target_start = df['datetime'].max() - pd.Timedelta(hours=2)
    plt.axvspan(target_start, df['datetime'].max(), color='orange', alpha=0.2, label='Black Ice Found')
    
    plt.title(f'Time Series: Air Temp vs Dew Point ({title_date})', fontsize=14)
    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Temperature (C)', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_time_series(df_analysis_10, '2026-02-10 Event')
plot_time_series(df_analysis_12, '2026-02-12 Event')

### 2. 상관관계 분석 (Correlation Heatmap)
밤사이 기온(ta), 습도(hm), 이슬점(td), 풍속(ws) 간에 어떠한 상관관계(Correlation)가 있는지 히트맵으로 시각화합니다. 피어슨 상관계수(Pearson Correlation)를 사용하여 수치가 1에 가까울수록 강한 양의 상관관계를, -1에 가까울수록 강한 음의 상관관계를 나타냅니다.

🔗 **관련 자료:** [히트맵(Heatmap)을 이용한 데이터 상관관계 분석](https://wikidocs.net/145891)

In [ ]:
if not df_analysis_10.empty:
    plt.figure(figsize=(8, 6))
    
    # 분석할 기상 컬럼 선택 및 피어슨 상관계수 행렬 계산
    corr_cols = ['ta', 'hm', 'td', 'ws']
    corr_matrix = df_analysis_10[corr_cols].corr()
    
    # 히트맵(Heatmap) 그리기
    sns.heatmap(corr_matrix, annot=True, cmap='vlag', vmin=-1, vmax=1, center=0, 
                fmt='.2f', square=True, linewidths=.5)
    
    plt.title('Correlation Heatmap (ta, hm, td, ws)', fontsize=14)
    plt.tight_layout()
    plt.show()

### 3. 2차원 커널 밀도 추정 (KDE Plot: 2D Density)
블랙아이스 발생 전날 밤부터 새벽 사이, 기온(ta)과 습도(hm)가 주로 **어느 구간에 가장 오래 머물렀는지(밀도, Density)** 를 확인합니다. 
색이 진한(밀도가 높은) 구간이 밤사이 유지된 주요 기상 조건입니다.

🔗 **관련 자료:** [커널 밀도 추정(KDE) 및 조인트 플롯(Jointplot) 이해하기](https://teddylee777.github.io/seaborn/seaborn-tutorial-04/)

In [ ]:
if not df_analysis_10.empty:
    # 조인트 플롯 (Jointplot) - KDE 방식
    g = sns.jointplot(data=df_analysis_10, x='ta', y='hm', kind='kde', 
                      cmap='Blues', fill=True, height=7, space=0)
    
    # 산점도를 위에 살짝 덮어그려 실제 데이터 분포 확인
    g.plot_joint(sns.scatterplot, color='gray', alpha=0.3, s=15, label='KMA Overnight Weather')
    
    # 블랙아이스 발생일 자정(00:00)부터 관측 시간(09:30)까지의 기상 궤적(Trajectory) 덧그리기
    mask_traj = (df_analysis_10['datetime'] >= '2026-02-10 00:00') & (df_analysis_10['datetime'] <= '2026-02-10 09:30')
    df_traj = df_analysis_10[mask_traj].sort_values('datetime')
    if not df_traj.empty:
        # 선으로 시간의 흐름 연결
        g.ax_joint.plot(df_traj['ta'], df_traj['hm'], color='red', linewidth=2, alpha=0.6, zorder=4, label='Trajectory (00:00~09:30)')
        # 궤적 위의 중간 지점들 (30분 간격)
        g.ax_joint.scatter(df_traj['ta'], df_traj['hm'], color='orange', s=50, edgecolor='black', zorder=5)
        
        # 시작점(자정)과 끝점(관측시점) 강조
        start_pt = df_traj.iloc[0]
        end_pt = df_traj.iloc[-1]
        g.ax_joint.scatter(start_pt['ta'], start_pt['hm'], color='yellow', marker='s', s=150, edgecolor='black', zorder=6, label='00:00 (Midnight)')
        g.ax_joint.scatter(end_pt['ta'], end_pt['hm'], color='red', marker='*', s=300, edgecolor='black', zorder=6, label='09:30 (Occurred)')
        
        # 텍스트 주석
        g.ax_joint.text(start_pt['ta']+0.1, start_pt['hm']+0.5, '00:00', color='black', weight='bold')
        g.ax_joint.text(end_pt['ta']-0.4, end_pt['hm']-1.5, '09:30', color='darkred', weight='bold')
    
    g.fig.suptitle('2D Density: Air Temp vs Humidity (Overnight) & Weather Trajectory', y=1.02, fontsize=14)
    g.set_axis_labels('Air Temperature (C)', 'Relative Humidity (%)', fontsize=12)
    g.ax_joint.legend()
    plt.show()

#### 💡 밤사이 기상 조건은 어떻게 블랙아이스를 만들었을까? (기상 궤적 분석)
단일 시점의 데이터만 보는 것이 아니라, **자정(00:00)부터 관측 시점(09:30)까지 기온과 습도가 어떻게 변해왔는지 궤적(Trajectory)** 을 붉은 선으로 연결했습니다.

1. **밤사이 응결 (00:00 ~ 새벽)**: 자정 무렵 기온이 영하로 크게 떨어지면서 상대습도가 가장 짙은 푸른색 구간(70~80%)을 맴돕니다. 바로 이 시간대가 공기 중의 수분이 도로 노면에 달라붙어 **블랙아이스(서리)를 형성하는 골든 타임**이었습니다.
2. **아침 관측 시점 (09:30, 빨간 별)**: 해가 뜬 아침 9시 이후가 되자, 기온이 미세하게 오르면서 궤적이 가파르게 아래(낮은 습도 방향)로 꺾이는 것을 볼 수 있습니다. 즉, 대기는 건조해졌지만 밤새 단단히 얼어붙은 얼음은 아직 도로에 남아있던 위험한 순간을 정확히 보여줍니다.

### 4. 이동 평균선 (Moving Average Trend)
기상 데이터는 바람 등 외부 요인에 의해 단기적으로 값이 들쭉날쭉할 수 있습니다. 이동 평균(Moving Average) 기법을 사용하면 노이즈를 제거하고 **온도가 떨어지는 근본적인 추세(Trend)**를 부드러운 곡선으로 시각화할 수 있습니다.

🔗 **관련 자료:** [판다스(Pandas) Rolling을 활용한 이동 평균선 구하기](https://wikidocs.net/152787)

In [ ]:
if not df_analysis_10.empty:
    df_trend = df_analysis_10.copy()
    
    # 2시간 단위 (30분 간격 데이터 4개) 이동 평균(Moving Average) 계산
    df_trend['ta_rolling_2h'] = df_trend['ta'].rolling(window=4, center=True).mean()
    
    plt.figure(figsize=(12, 5))
    
    # 원래 데이터(노이즈 포함)를 흐리게 표시
    sns.scatterplot(data=df_trend, x='datetime', y='ta', color='gray', alpha=0.4, label='Raw Data (30min interval)')
    
    # 이동 평균선 (추세) 표시
    sns.lineplot(data=df_trend, x='datetime', y='ta_rolling_2h', color='darkred', linewidth=3, label='2-Hour Moving Average')
    
    plt.axhline(0, color='blue', linestyle='--', alpha=0.3, label='Freezing Point (0C)')
    
    plt.title('Temperature Trend Smoothing (Moving Average)', fontsize=14)
    plt.xlabel('Time', fontsize=12)
    plt.ylabel('Air Temperature (C)', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 💡 사전 조건 종합 분석 결론
전날 오후부터 블랙아이스 발생 당일 아침까지의 데이터를 다양한 기법으로 분석한 결과, **블랙아이스가 생성되기 위한 3가지 사전 골든 조건**이 성립함을 증명했습니다.

1. **큰 일교차 (이동평균 및 시계열)**: 전날 오후 0도를 웃돌던 기온이 밤사이에 급격히 0도 이하로 곤두박질치는 추세가 나타납니다.
2. **기온과 습도의 상관관계 (히트맵 & KDE)**: 기온이 떨어지는 새벽일수록 대기의 상대습도는 70~80%로 높게 머물러 있습니다. (차가운 공기는 수분을 적게 포함하므로 상대습도가 높아짐)
3. **이슬점의 근접성 (시계열 교차)**: 대기 기온(ta)이 이슬점 온도(td)와 거의 맞닿을 만큼 가까워집니다. 이 시점에 대기 중의 넉넉한 습기(수분)가 차갑게 얼어붙은 도로 노면에 닿으며 이슬 대신 서리 형태의 얇은 얼음(블랙아이스)을 형성하게 됩니다.